# 🧴 Skincare Recommendation System
**Features:** Low-sell warnings · Seasonal picks · Expiry offers · Click-based surfacing

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import json
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Import our engine
import sys; sys.path.insert(0, ".")
from recommendation_engine import (
    load_data, get_low_selling_products, get_seasonal_recommendations,
    get_near_expiry_offers, get_top_clicked_products,
    init_click_db, record_click, build_recommendation_payload
)

df = load_data("Skin_Care.csv")
init_click_db()
print(f"✅ {len(df)} products loaded — columns: {list(df.columns)}")


## 1️⃣ Dataset Overview

In [ ]:
display(df[["product_name","brand","product_type","price",
              "peak_season","low_sell_probability","days_to_expiry"]].head(10))
print(f"
Product types: {df.product_type.value_counts().to_dict()}")
print(f"Seasons:
{df.peak_season.value_counts().to_dict()}")
print(f"
Expiry range: {df.days_to_expiry.min():.0f} – {df.days_to_expiry.max():.0f} days")


## 2️⃣ Low-Selling Products → Free Sample Campaign

In [ ]:
low = get_low_selling_products(df)
print(f"⚠️  {len(low)} products flagged as slow-moving (prob ≥ 0.35)")
display(low[["product_name","brand","product_type","price",
            "low_sell_probability","warning_message","badge"]].head(8))


In [ ]:
# Visualise low-sell probability distribution
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["low_sell_probability"].hist(bins=20, ax=axes[0], color="#FF6B6B", edgecolor="white")
axes[0].axvline(0.35, color="red", linestyle="--", label="Free-sample threshold")
axes[0].set_title("Low-Sell Probability Distribution")
axes[0].set_xlabel("Probability"); axes[0].set_ylabel("# Products")
axes[0].legend()

low_by_type = low.groupby("product_type").size().sort_values(ascending=False)
low_by_type.plot(kind="bar", ax=axes[1], color="#FFD166", edgecolor="white")
axes[1].set_title("Low-Sellers by Product Type")
axes[1].set_xlabel(""); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()


## 3️⃣ Seasonal Recommendations

In [ ]:
for season in ["rainy", "dry", "stable"]:
    recs = get_seasonal_recommendations(df, season=season, top_n=5)
    print(f"
🌦️  Season: {season.upper()}  →  {len(recs)} recs")
    display(recs[["product_name","brand","product_type","price",
                  "peak_season","recommended_badge"]].head(5))


## 4️⃣ Near-Expiry Offers — Buy 2 Get 3 Free

In [ ]:
offers = get_near_expiry_offers(df)
print(f"⏰  {len(offers)} products near expiry (within 365 days)")
display(offers[["product_name","brand","price",
               "days_to_expiry","urgency_level",
               "offer_badge","offer_detail"]].head(10))


In [ ]:
# Urgency breakdown chart
fig, ax = plt.subplots(figsize=(7, 4))
urgency_counts = offers["urgency_level"].value_counts()
colors = ["#FF4757", "#FFA502", "#2ED573"]
urgency_counts.plot(kind="bar", ax=ax, color=colors[:len(urgency_counts)], edgecolor="white")
ax.set_title("Near-Expiry Products by Urgency")
ax.set_xlabel(""); ax.tick_params(axis="x", rotation=20)
for i, v in enumerate(urgency_counts):
    ax.text(i, v + 0.3, str(v), ha="center", fontweight="bold")
plt.tight_layout(); plt.show()


## 5️⃣ Click Tracking & Top-Clicked Surfacing

In [ ]:
# Simulate a user clicking a product 3 times
session = "notebook_demo"
target_product = df.iloc[5]

for _ in range(3):
    record_click(session, target_product["product_id"], target_product["product_name"])
    
hot = get_top_clicked_products(df, session_id=session)
if len(hot):
    print(f"🔥 Hot products for session {session!r}:")
    display(hot[["product_name","brand","click_count","reminder_banner"]])
else:
    print("No hot products yet for this session.")


## 6️⃣ Full Recommendation Payload

In [ ]:
payload = build_recommendation_payload(df, session_id="notebook_demo")

print("📦 Payload summary:")
for k, v in payload.items():
    if isinstance(v, list):
        print(f"  {k}: {len(v)} items")
    else:
        print(f"  {k}: {v}")
        
# Save to file for API / React consumption
with open("recommendation_payload.json", "w") as f:
    json.dump(payload, f, indent=2, default=str)
print("
💾 Saved → recommendation_payload.json")


## 7️⃣ SQLite Analytics Dashboard

In [ ]:
con = sqlite3.connect("skincare_clicks.db")

# Load products into SQL for query demos
df_sql = df[["product_id","product_name","brand","product_type",
             "price","price_num","low_sell_probability","days_to_expiry",
             "expiry_date","peak_season"]].copy()
df_sql["expiry_date"] = df_sql["expiry_date"].astype(str)
df_sql.to_sql("products", con, if_exists="replace", index=False)

print("SQL — Low-selling products:")
display(pd.read_sql("SELECT product_name, brand, low_sell_probability FROM products WHERE low_sell_probability >= 0.35 ORDER BY low_sell_probability DESC LIMIT 5", con))

print("
SQL — Near-expiry offers:")
display(pd.read_sql("SELECT product_name, brand, days_to_expiry FROM products WHERE days_to_expiry BETWEEN 0 AND 365 ORDER BY days_to_expiry ASC LIMIT 5", con))

print("
SQL — Click leaderboard:")
display(pd.read_sql("SELECT product_id, COUNT(*) as total_clicks FROM product_clicks GROUP BY product_id ORDER BY total_clicks DESC LIMIT 5", con))

con.close()
